Calculate the sum of a vector with OpenCL.

In [ ]:
!pip install pyopencl

In [1]:
import numpy as np
import pyopencl as cl
import pyopencl.array as cl_array

In [2]:
ctx = cl.create_some_context()
queue = cl.CommandQueue(ctx)

In [3]:
platform = cl.get_platforms()[0]
device = platform.get_devices()[0]
print("Maximum work group size:", device.max_work_group_size)

Maximum work group size: 256


In the following, we wish to divide the data in smaller workgroups. For this purpose, we will define the workgroup size and the number of data points. The implementation in the following is only valid for a workgroup size that is a divisor of the number of data points. If this is not the case, one needs to pad the data array with dummy variables.

In [4]:
workgroup_size = 2**10
n_workgroups = 4
vector_size = workgroup_size * n_workgroups
print("Workgroup size:", workgroup_size)
print("Vector size:", vector_size)

Workgroup size: 1024
Vector size: 4096


Calculating the sum of a vector is a recursive process and, therefore, not well suited for a SIMD architecture. Different approaches can be used to sum a vector. Because of the desing with workgroups, one can implement a kernel that sums the part of the vector that is assigned to that workgroup. Then, instead of summing these partial sums on the device, it is often more efficient to perform this final step on the CPU.

The first option is to let one thread sum all values in a workgroup. This is slow because no parallelism is used.

The second option is to implement a binary tree, which has parallelism. The implementation below only works for a work group size that is a power of two. Since if-else statements are slow on the GPU, in case of a workgroup that is not a power of two, it is better to pad the input vector with zero elements until reaching a power of two. Also, the barrier is necessary to avoid race conditions between threads.

In [5]:
kernel = """
__kernel void sum1(__global const long int *vec,
                   __global long int *partial_sums)
{
  int group_size = get_local_size(0);
  int local_id = get_local_id(0);
  int group_id = get_group_id(0);
  int global_id = get_global_id(0);

  if (local_id == 0){
    long int sum = 0;
    for(int i = 0; i < group_size; i++){
      sum += vec[global_id + i];
    }
    partial_sums[group_id] = sum;
  }
}
__kernel void sum2(__global long int *vec,
                   __global long int *partial_sums)
{
  int group_size = get_local_size(0);
  int local_id = get_local_id(0);
  int group_id = get_group_id(0);
  int global_id = get_global_id(0);

  int step = 2;
  while (step <= group_size) {
    if (local_id % step == 0) {
      vec[global_id] += vec[global_id + step / 2];
    }
    barrier(CLK_GLOBAL_MEM_FENCE);
    step *= 2;
  }
  if (local_id == 0){
    partial_sums[group_id] = vec[global_id];
  }
}
"""

In [6]:
prg = cl.Program(ctx, kernel).build()

Let us define a vector with *n* values from zero to *n-1* and calculate the sum.

The kernel calculates the partial sums within the workgroup. The final sum will be calculated on the CPU, so we need to create an output array with a size of the number of workgroups.

The input variables of the program are as follows:
 1. the command queue
 1. the global size of the data array, possibly multi-dimensional
 1. the size of the workgroups, which needs to divide the global size evenly
 1. the buffers for the input variables of the kernel function

Notice that we use an ```np.int64``` for the vector and a ```long int``` in the kernel to prevent an integer overflow.

In [7]:
np_vector = np.arange(vector_size, dtype=np.int64)
cl_vector = cl_array.to_device(queue, np_vector)

cl_partial_sums1 = cl_array.empty(queue, n_workgroups, dtype=np.int64)
cl_partial_sums2 = cl_array.empty(queue, n_workgroups, dtype=np.int64)

In [ ]:
event = prg.sum1(queue,
                 (vector_size,),
                 (workgroup_size,),
                 cl_vector.data,
                 cl_partial_sums1.data
                )

In [ ]:
event = prg.sum2(queue,
                 (vector_size,),
                 (workgroup_size,),
                 cl_vector.data,
                 cl_partial_sums2.data
                )

In [ ]:
print(cl_partial_sums1)

In [ ]:
np_partial_sums1 = cl_partial_sums1.get()
vector_sum1 = np.sum(np_partial_sums1)
np_partial_sums2 = cl_partial_sums2.get()
vector_sum2 = np.sum(np_partial_sums2)

In [ ]:
print("The sum calculated by OpenCL:", vector_sum1)
print("The sum calculated by OpenCL:", vector_sum2)
print("The exact value of the sum:  ", vector_size*(vector_size-1)//2)